In [1]:
"""
=========================================================
04_hyperparameter_tuning.py
=========================================================

Purpose
-------
Tunes CatBoost hyperparameters using Optuna.

Input
-----
data/processed/feature_engineered.csv
outputs/selected_features.json

Output
------
outputs/best_hyperparameters.json
outputs/optuna_trials.csv
"""

'\n=========================================================\n04_hyperparameter_tuning.py\n=========================================================\n\nPurpose\n-------\nTunes CatBoost hyperparameters using Optuna.\n\nInput\n-----\ndata/processed/feature_engineered.csv\noutputs/selected_features.json\n\nOutput\n------\noutputs/best_hyperparameters.json\noutputs/optuna_trials.csv\n'

In [2]:
import json
import warnings

import optuna
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from pathlib import Path
warnings.filterwarnings("ignore")

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

INPUT_FILE = BASE_DIR /  "data" / "feature_engineered.csv"
FIGURE_PATH = BASE_DIR /"outputs"

In [4]:
#Global declare random state
RANDOM_STATE = 42

In [5]:
# ==========================================================
# LOAD DATA
# ==========================================================

print("=" * 60)
print("Loading Dataset")
print("=" * 60)

df = pd.read_csv(INPUT_FILE)

with open(f"{FIGURE_PATH}/selected_features.json") as f:
    FEATURES = json.load(f)

TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

Loading Dataset


In [6]:
# ==========================================================
# DEFINE CATEGORICAL FEATURES
# ==========================================================

categorical_features = [
    "Category_Bucket_final",
    "Vertical",
    "self_dealer_status",
    "Dealing_Zone",
    "plan",
]

cat_features = [
    X.columns.get_loc(col)
    for col in categorical_features
    if col in X.columns
]

# Convert categorical columns to string

for col in categorical_features:
    if col in X.columns:
        X[col] = X[col].fillna("Missing").astype(str)

In [7]:
# ==========================================================
# TRAIN / VALIDATION SPLIT
# ==========================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

In [8]:
# ==========================================================
# OPTUNA OBJECTIVE
# ==========================================================

def objective(trial):

    params = {

        "loss_function": "Logloss",

        "eval_metric": "AUC",

        "iterations": trial.suggest_int(
            "iterations",
            200,
            600
        ),

        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg",
            1,
            10
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            5
        ),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            5
        ),

        "border_count": trial.suggest_int(
            "border_count",
            64,
            255
        ),

        "verbose": False,

        "random_seed": RANDOM_STATE

    }

    model = CatBoostClassifier(**params)

    model.fit(

        X_train,
        y_train,

        cat_features=cat_features,

        eval_set=(X_valid, y_valid),

        use_best_model=True,

        verbose=False

    )

    pred = model.predict_proba(X_valid)[:, 1]

    auc = roc_auc_score(
        y_valid,
        pred
    )

    return auc

In [9]:
# ==========================================================
# RUN OPTUNA
# ==========================================================

print("\nStarting Hyperparameter Search...\n")

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-07-11 22:25:50,536] A new study created in memory with name: no-name-afd63cc8-6d34-4c90-baac-180554db229a



Starting Hyperparameter Search...



Best trial: 0. Best value: 0.602075:   3%|▎         | 1/30 [06:12<2:59:54, 372.22s/it]

[I 2026-07-11 22:32:02,743] Trial 0 finished with value: 0.6020749318493273 and parameters: {'iterations': 475, 'depth': 7, 'learning_rate': 0.09198342662322365, 'l2_leaf_reg': 4, 'random_strength': 2.671842250443603, 'bagging_temperature': 1.1477829118517184, 'border_count': 113}. Best is trial 0 with value: 0.6020749318493273.


Best trial: 0. Best value: 0.602075:   7%|▋         | 2/30 [14:34<3:29:29, 448.92s/it]

...

[I 2026-07-12 00:24:15,705] Trial 27 finished with value: 0.6015396687806838 and parameters: {'iterations': 577, 'depth': 4, 'learning_rate': 0.05930510064559312, 'l2_leaf_reg': 5, 'random_strength': 1.9777329117921112, 'bagging_temperature': 4.287723470741804, 'border_count': 163}. Best is trial 23 with value: 0.6053343053315714.


Best trial: 23. Best value: 0.605334:  97%|█████████▋| 29/30 [2:02:31<03:29, 209.14s/it]

[I 2026-07-12 00:28:22,102] Trial 28 finished with value: 0.598788336297536 and parameters: {'iterations': 599, 'depth': 5, 'learning_rate': 0.10417745214898189, 'l2_leaf_reg': 7, 'random_strength': 0.9038876427765721, 'bagging_temperature': 3.3615769304934857, 'border_count': 134}. Best is trial 23 with value: 0.6053343053315714.


Best trial: 23. Best value: 0.605334: 100%|██████████| 30/30 [2:05:22<00:00, 250.75s/it]


[I 2026-07-12 00:31:13,052] Trial 29 finished with value: 0.6028913592627283 and parameters: {'iterations': 549, 'depth': 4, 'learning_rate': 0.035003182594457206, 'l2_leaf_reg': 4, 'random_strength': 2.7359822871908226, 'bagging_temperature': 4.483355770465184, 'border_count': 110}. Best is trial 23 with value: 0.6053343053315714.


In [11]:
# ==========================================================
# SAVE RESULTS
# ==========================================================

best_params = study.best_params

with open(
    f"{FIGURE_PATH}/best_hyperparameters.json",
    "w"
) as f:

    json.dump(
        best_params,
        f,
        indent=4
    )

trials = study.trials_dataframe()

trials.to_csv(
    f"{FIGURE_PATH}/optuna_trials.csv",
    index=False
)

In [12]:
# ==========================================================
# SUMMARY
# ==========================================================

print("\n" + "=" * 60)
print("Hyperparameter Tuning Complete")
print("=" * 60)

print("\nBest ROC AUC")
print(study.best_value)

print("\nBest Parameters")

for k, v in best_params.items():
    print(f"{k:25} : {v}")

print("\nSaved Files")

print(f"{FIGURE_PATH}/best_hyperparameters.json")
print(f"{FIGURE_PATH}/optuna_trials.csv")


Hyperparameter Tuning Complete

Best ROC AUC
0.6053343053315714

Best Parameters
iterations                : 307
depth                     : 4
learning_rate             : 0.13584344074104596
l2_leaf_reg               : 6
random_strength           : 3.6944793157690925
bagging_temperature       : 3.989916476550814
border_count              : 150

Saved Files
e:\github\AB Testing\outputs/best_hyperparameters.json
e:\github\AB Testing\outputs/optuna_trials.csv
